
# 02 — Popularity-Based Movie Recommender

## Objective
Recommend the top K movies using a weighted rating score that considers
both the average rating and the number of ratings.

## Method
Use an IMDb-style weighted rating formula to reduce the influence of
movies with very few ratings.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:

movies_clean = pd.read_csv("/content/movies_clean.csv")

print("Dataset shape:", movies_clean.shape)
print("\nColumns:")
print(movies_clean.columns.tolist())

display(movies_clean.head())


Dataset shape: (12794, 8)

Columns:
['genres', 'overview', 'release_date', 'title', 'release_year', 'tmdbId', 'avg_rating', 'rating_count']


,genres,overview,release_date,title,release_year,tmdbId,avg_rating,rating_count
0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",An atmospheric coming-of-age story featuring a...,2010-05-25,Venice,2010,79782,2.977612,67.0
1,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","The town of Derry has a secret, but no one tol...",2013-10-12,The Sleepover,2013,141210,2.953488,43.0
2,"[{'id': 18, 'name': 'Drama'}]","As her surroundings are invaded by outsiders, ...",2012-06-20,The Farmer's Wife,2012,143750,2.891304,46.0
3,"[{'id': 99, 'name': 'Documentary'}]","Using personal stories, this powerful document...",2012-03-22,A Place at the Table,2012,84198,2.862500,80.0
4,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...",She's as hot as Britney Spears. Hotter. She pa...,2012-10-09,Romance,2012,171982,2.789007,564.0



## 1. Inspect Rating Statistics

Before calculating popularity scores, inspect the average ratings and
rating counts to understand the available data and identify missing values.


In [3]:

print("Missing values:")
print(movies_clean[["avg_rating", "rating_count"]].isna().sum())

print("\nSummary statistics:")
display(movies_clean[["avg_rating", "rating_count"]].describe())

print("\nMovies without ratings:",
      movies_clean["avg_rating"].isna().sum())


Missing values:
avg_rating      142
rating_count    142
dtype: int64

Summary statistics:


,avg_rating,rating_count
count,12652.000000,12652.000000
mean,3.054345,151.226446
std,0.761786,866.744963
min,0.500000,1.000000
25%,2.666667,2.000000
50%,3.158947,6.000000
75%,3.500000,25.000000
max,5.000000,35297.000000



Movies without ratings: 142


In [4]:

# Consider only movies that have rating statistics.
rated_movies = movies_clean.dropna(
    subset=["avg_rating", "rating_count"]
).copy()

# C: mean average rating across rated movies.
C = rated_movies["avg_rating"].mean()

# m: 70th percentile of rating counts.
m = rated_movies["rating_count"].quantile(0.70)

print(f"Number of rated movies: {len(rated_movies)}")
print(f"Overall mean rating (C): {C:.4f}")
print(f"70th percentile of rating count (m): {m:.2f}")


Number of rated movies: 12652
Overall mean rating (C): 3.0543
70th percentile of rating count (m): 17.00



## 2. Calculate Weighted Ratings

Calculate a score that combines each movie's average rating and rating count.
Movies with fewer ratings are pulled closer to the overall average rating,
reducing the influence of small rating samples.


In [5]:

rated_movies["weighted_rating"] = (
    (rated_movies["rating_count"] / (rated_movies["rating_count"] + m))
    * rated_movies["avg_rating"]
    +
    (m / (rated_movies["rating_count"] + m))
    * C
)

print("Weighted ratings calculated.")
print("\nFirst 5 movies:")
display(
    rated_movies[
        ["title", "avg_rating", "rating_count", "weighted_rating"]
    ].head()
)


Weighted ratings calculated.

First 5 movies:


,title,avg_rating,rating_count,weighted_rating
0,Venice,2.977612,67.0,2.993141
1,The Sleepover,2.953488,43.0,2.982064
2,The Farmer's Wife,2.891304,46.0,2.935299
3,A Place at the Table,2.862500,80.0,2.896122
4,Romance,2.789007,564.0,2.796771



## 3. Generate Popularity-Based Recommendations

Sort movies in descending order of weighted rating.
Movies with the highest weighted scores will appear first.


In [6]:

# Sort movies from highest to lowest weighted rating.
popularity_ranking = rated_movies.sort_values(
    by="weighted_rating",
    ascending=False
).reset_index(drop=True)

# Display the top 10 movies.
top_10_movies = popularity_ranking.head(10)

display(
    top_10_movies[
        [
            "title",
            "release_year",
            "avg_rating",
            "rating_count",
            "weighted_rating"
        ]
    ]
)


,title,release_year,avg_rating,rating_count,weighted_rating
0,Inception,2010,4.161756,35297.0,4.161223
1,Planet Earth II,2016,4.352632,95.0,4.155570
2,Whiplash,2013,4.226776,183.0,4.127119
3,The Intouchables,2011,4.122234,11253.0,4.120623
4,Whiplash,2014,4.120284,8455.0,4.118145
5,Piper,2016,4.129767,944.0,4.110743
6,Over the Garden Wall,2014,4.219745,157.0,4.105884
7,Wild Tales,2014,4.110053,1522.0,4.098391
8,Paperman,2012,4.095850,2024.0,4.087175
9,Interstellar,2014,4.086824,18693.0,4.085886



## 4. Create a Reusable Recommendation Function

Create a function that accepts K and returns the top K movies
ranked by weighted rating.


In [7]:

def recommend_popular_movies(k=10):
    """
    Return the top K movies ranked by weighted rating.

    Parameters:
        k (int): Number of movies to recommend.

    Returns:
        DataFrame containing the top K movies.
    """
    if k <= 0:
        raise ValueError("k must be a positive integer.")

    return popularity_ranking.head(k)[
        [
            "tmdbId",
            "title",
            "release_year",
            "genres",
            "overview",
            "avg_rating",
            "rating_count",
            "weighted_rating"
        ]
    ]

# Example: recommend the top 10 movies.
display(recommend_popular_movies(10))


,tmdbId,title,release_year,genres,overview,avg_rating,rating_count,weighted_rating
0,27205,Inception,2010,"[{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...","Cobb, a skilled thief who commits corporate es...",4.161756,35297.0,4.161223
1,420714,Planet Earth II,2016,"[{'id': 99, 'name': 'Documentary'}]",David Attenborough presents a documentary seri...,4.352632,95.0,4.155570
2,367412,Whiplash,2013,"[{'id': 18, 'name': 'Drama'}, {'id': 10402, 'n...","A ferocious, bullying music teacher teaches a ...",4.226776,183.0,4.127119
3,77338,The Intouchables,2011,"[{'id': 18, 'name': 'Drama'}, {'id': 35, 'name...",A true story of two men who should never have ...,4.122234,11253.0,4.120623
4,244786,Whiplash,2014,"[{'id': 18, 'name': 'Drama'}]","Under the direction of a ruthless instructor, ...",4.120284,8455.0,4.118145
5,399106,Piper,2016,"[{'id': 10751, 'name': 'Family'}, {'id': 16, '...",A mother bird tries to teach her little one ho...,4.129767,944.0,4.110743
6,409696,Over the Garden Wall,2014,"[{'id': 12, 'name': 'Adventure'}, {'id': 16, '...",Two brothers find themselves lost in a mysteri...,4.219745,157.0,4.105884
7,265195,Wild Tales,2014,"[{'id': 18, 'name': 'Drama'}, {'id': 53, 'name...",Six short stories that explore the extremities...,4.110053,1522.0,4.098391
8,140420,Paperman,2012,"[{'id': 16, 'name': 'Animation'}, {'id': 10751...",An urban office worker finds that paper airpla...,4.095850,2024.0,4.087175
9,157336,Interstellar,2014,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...",Interstellar chronicles the adventures of a gr...,4.086824,18693.0,4.085886


In [10]:

# Check that scores are sorted in descending order.
scores = popularity_ranking["weighted_rating"]

print("Scores sorted correctly:", scores.is_monotonic_decreasing)
print("Number of ranked movies:", len(popularity_ranking))
print("Unique TMDB IDs:", popularity_ranking["tmdbId"].nunique())

# Check that the function returns exactly K movies.
test_k = 15
recommendations = recommend_popular_movies(test_k)

print(f"\nRequested K: {test_k}")
print("Movies returned:", len(recommendations))

display(
    recommendations[
        ["title", "avg_rating", "rating_count", "weighted_rating"]
    ]
)


Scores sorted correctly: True
Number of ranked movies: 12652
Unique TMDB IDs: 12652

Requested K: 15
Movies returned: 15


,title,avg_rating,rating_count,weighted_rating
0,Inception,4.161756,35297.0,4.161223
1,Planet Earth II,4.352632,95.0,4.155570
2,Whiplash,4.226776,183.0,4.127119
3,The Intouchables,4.122234,11253.0,4.120623
4,Whiplash,4.120284,8455.0,4.118145
5,Piper,4.129767,944.0,4.110743
6,Over the Garden Wall,4.219745,157.0,4.105884
7,Wild Tales,4.110053,1522.0,4.098391
8,Paperman,4.095850,2024.0,4.087175
9,Interstellar,4.086824,18693.0,4.085886
